# POS/Cash Balance Data Cleaning

This notebook cleans the monthly POS/cash records using the problems found during EDA. It keeps records linked to the project applicants, groups rare contract statuses, flags unusual installment and delinquency values, and removes features with too many missing values.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
raw_path = project_root / "data" / "raw" / "POS_CASH_balance.csv"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "interim" / "pos_cash_balance_clean.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [raw_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Raw input:", raw_path)
print("Clean output:", output_path)

Raw input: /Users/taranveersingh/A-MRP/data/raw/POS_CASH_balance.csv
Clean output: /Users/taranveersingh/A-MRP/data/interim/pos_cash_balance_clean.pkl


## Load data and retain project applicants


In [7]:
pos_raw = pd.read_csv(raw_path)
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
project_ids = training_id_set.union(set(test_ids))
original_rows, original_columns = pos_raw.shape

pos_clean = pos_raw.loc[pos_raw["SK_ID_CURR"].isin(project_ids)].copy().reset_index(drop=True)
del pos_raw
out_of_scope_rows = original_rows - len(pos_clean)
print("Raw rows:", original_rows)
print("Project rows retained:", len(pos_clean))
print("Out-of-scope rows removed:", out_of_scope_rows)
print("Previous loans represented:", pos_clean["SK_ID_PREV"].nunique())
print("Applicants represented:", pos_clean["SK_ID_CURR"].nunique())

Raw rows: 10001358
Project rows retained: 8543375
Out-of-scope rows removed: 1457983
Previous loans represented: 800337
Applicants represented: 289444


The rows that were removed belong to applicants outside the project's training and test data.


## Validate identifiers and monthly keys


In [10]:
missing_current_ids = int(pos_clean["SK_ID_CURR"].isna().sum())
missing_previous_ids = int(pos_clean["SK_ID_PREV"].isna().sum())
exact_duplicates = int(pos_clean.duplicated().sum())
duplicate_month_keys = int(pos_clean.duplicated(["SK_ID_PREV", "MONTHS_BALANCE"]).sum())
if exact_duplicates > 0:
    pos_clean = pos_clean.drop_duplicates().reset_index(drop=True)
assert missing_current_ids == 0 and missing_previous_ids == 0
assert duplicate_month_keys == exact_duplicates, "Conflicting records exist for the same loan and month."
print("Missing applicant IDs:", missing_current_ids)
print("Missing previous-loan IDs:", missing_previous_ids)
print("Exact duplicate rows removed:", exact_duplicates)
print("Conflicting loan-month records:", duplicate_month_keys - exact_duplicates)

Missing applicant IDs: 0
Missing previous-loan IDs: 0
Exact duplicate rows removed: 0
Conflicting loan-month records: 0


The IDs are clean, no missing values, no duplicates, no conflicting records for the same loan-month.


## Standardize contract status


In [13]:
pos_clean["NAME_CONTRACT_STATUS"] = pos_clean["NAME_CONTRACT_STATUS"].str.strip().fillna("Unknown")
training_mask = pos_clean["SK_ID_CURR"].isin(training_id_set)
status_counts = pos_clean.loc[training_mask, "NAME_CONTRACT_STATUS"].value_counts()
rare_statuses = status_counts[status_counts < 100].index
rare_status_mask = pos_clean["NAME_CONTRACT_STATUS"].isin(rare_statuses)
pos_clean.loc[rare_status_mask, "NAME_CONTRACT_STATUS"] = "Other rare"
print("Rare statuses consolidated:", list(rare_statuses))
pos_clean["NAME_CONTRACT_STATUS"].value_counts()

Rare statuses consolidated: ['Canceled', 'XNA']


NAME_CONTRACT_STATUS
Active                   7818577
Completed                 634872
Signed                     74625
Demand                      6110
Returned to the store       4591
Approved                    4221
Amortized debt               365
Other rare                    14
Name: count, dtype: int64

Canceled and XNA statuses are very rare here, so they get grouped into an Other rare category. Active and Completed still make up almost all of the records.


## Audit month, installment and delinquency values


In [16]:
future_month = pos_clean["MONTHS_BALANCE"].gt(0)
extreme_month = pos_clean["MONTHS_BALANCE"].lt(-1200)
negative_total_installments = pos_clean["CNT_INSTALMENT"].lt(0)
negative_future_installments = pos_clean["CNT_INSTALMENT_FUTURE"].lt(0)
future_above_total = pos_clean["CNT_INSTALMENT_FUTURE"].gt(pos_clean["CNT_INSTALMENT"])
negative_dpd = pos_clean["SK_DPD"].lt(0) | pos_clean["SK_DPD_DEF"].lt(0)
dpd_definition_above_dpd = pos_clean["SK_DPD_DEF"].gt(pos_clean["SK_DPD"])
completed_with_remaining = (
    pos_clean["NAME_CONTRACT_STATUS"].eq("Completed")
    & pos_clean["CNT_INSTALMENT_FUTURE"].gt(0)
)
active_with_zero_future = (
    pos_clean["NAME_CONTRACT_STATUS"].eq("Active")
    & pos_clean["CNT_INSTALMENT_FUTURE"].eq(0)
)
schedule_missing = pos_clean[["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]].isna().any(axis=1)

pos_clean["POS_MONTH_ANOMALY"] = (future_month | extreme_month).astype("int8")
pos_clean["POS_SCHEDULE_MISSING"] = schedule_missing.astype("int8")
pos_clean["POS_FUTURE_ABOVE_TOTAL"] = future_above_total.fillna(False).astype("int8")
pos_clean["POS_DPD_INCONSISTENCY"] = dpd_definition_above_dpd.astype("int8")
pos_clean["POS_COMPLETED_WITH_REMAINING"] = completed_with_remaining.fillna(False).astype("int8")
pos_clean["POS_ACTIVE_WITH_ZERO_FUTURE"] = active_with_zero_future.fillna(False).astype("int8")
pos_clean.loc[future_month | extreme_month, "MONTHS_BALANCE"] = np.nan
pos_clean.loc[negative_total_installments, "CNT_INSTALMENT"] = np.nan
pos_clean.loc[negative_future_installments, "CNT_INSTALMENT_FUTURE"] = np.nan
pos_clean.loc[negative_dpd, ["SK_DPD", "SK_DPD_DEF"]] = np.nan
print("Future month values corrected:", int(future_month.sum()))
print("Extreme month values corrected:", int(extreme_month.sum()))
print("Negative installment-count values corrected:", int((negative_total_installments | negative_future_installments).sum()))
print("Negative delinquency values corrected:", int(negative_dpd.sum()))
print("Future installments above total records:", int(future_above_total.sum()))
print("Completed contracts with installments remaining:", int(completed_with_remaining.sum()))
print("Active contracts with zero future installments:", int(active_with_zero_future.sum()))
print("Missing repayment schedules:", int(schedule_missing.sum()))

Future month values corrected: 0
Extreme month values corrected: 0
Negative installment-count values corrected: 0
Negative delinquency values corrected: 0
Future installments above total records: 6430
Completed contracts with installments remaining: 3
Active contracts with zero future installments: 379838
Missing repayment schedules: 21963


No future or extremely old month values, negative installment counts, or negative delinquency days were found. A fair number of records show more future installments than total installments, or an active contract with zero future installments left, so these are kept and flagged for review instead of being removed.


## Build training-only feature decisions


In [19]:
MISSING_THRESHOLD = 0.50
training_pos = pos_clean.loc[pos_clean["SK_ID_CURR"].isin(training_id_set)]
decision_rows = []
for column in pos_clean.columns:
    if column in ["SK_ID_CURR", "SK_ID_PREV"]:
        continue
    series = training_pos[column]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    decision = "Keep"
    reason = "Retain for applicant-level aggregation and later target-based selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-linked missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant among training-linked monthly records"
    decision_rows.append({
        "feature": column, "data_type": str(series.dtype),
        "missing_count": int(series.isna().sum()), "missing_rate": missing_rate,
        "unique_non_missing": int(unique_non_missing), "decision": decision,
        "reason": reason, "target_association_stage": "After applicant-level aggregation"
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "missing_rate"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[
    feature_decisions["decision"] == "Remove", "feature"
] .tolist()
pos_clean = pos_clean.drop(columns=removed_features)
print("Features removed:", removed_features)
feature_decisions.round(5)

Features removed: ['POS_MONTH_ANOMALY', 'POS_DPD_INCONSISTENCY']


,feature,data_type,missing_count,missing_rate,unique_non_missing,decision,reason,target_association_stage
0,CNT_INSTALMENT,float64,17507,0.00256,70,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
1,CNT_INSTALMENT_FUTURE,float64,17504,0.00256,78,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
2,MONTHS_BALANCE,float64,0,0.00000,96,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
3,NAME_CONTRACT_STATUS,str,0,0.00000,8,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
4,SK_DPD,float64,0,0.00000,3317,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
5,SK_DPD_DEF,float64,0,0.00000,1971,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
6,POS_SCHEDULE_MISSING,int8,0,0.00000,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
7,POS_FUTURE_ABOVE_TOTAL,int8,0,0.00000,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
8,POS_COMPLETED_WITH_REMAINING,int8,0,0.00000,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
9,POS_ACTIVE_WITH_ZERO_FUTURE,int8,0,0.00000,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation


Two features were removed: the month-anomaly and DPD-inconsistency flags created earlier in this notebook. Since no future/extreme months or DPD inconsistencies were actually found, both ended up constant.


## Fill categorical missingness and create missingness features


In [22]:
retained_categorical = pos_clean.select_dtypes(exclude="number").columns.tolist()
categorical_missing_before = int(pos_clean[retained_categorical].isna().sum().sum())
pos_clean[retained_categorical] = pos_clean[retained_categorical].fillna("Unknown")
record_features = [c for c in pos_clean.columns if c not in ["SK_ID_CURR", "SK_ID_PREV"]]
pos_clean["POS_RECORD_MISSING_COUNT"] = pos_clean[record_features].isna().sum(axis=1).astype("int8")
pos_clean["POS_RECORD_MISSING_RATE"] = pos_clean["POS_RECORD_MISSING_COUNT"] / len(record_features)
print("Categorical missing values filled:", categorical_missing_before)
print(pos_clean[["POS_RECORD_MISSING_COUNT", "POS_RECORD_MISSING_RATE"]].describe().round(5))

Categorical missing values filled: 0
       POS_RECORD_MISSING_COUNT  POS_RECORD_MISSING_RATE
count              8.543375e+06             8.543375e+06
mean               5.120000e-03             5.100000e-04
std                1.009500e-01             1.010000e-02
min                0.000000e+00             0.000000e+00
25%                0.000000e+00             0.000000e+00
50%                0.000000e+00             0.000000e+00
75%                0.000000e+00             0.000000e+00
max                2.000000e+00             2.000000e-01


This column tracks how much information is missing for each monthly record. The missing rate here is very low, well under 1%.


## Validate the cleaned table


In [25]:
numeric_columns = pos_clean.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(pos_clean[c].dropna()).sum()) for c in numeric_columns)
validation_checks = pd.DataFrame([
    {"check": "Only project applicants retained", "passed": set(pos_clean["SK_ID_CURR"]).issubset(project_ids)},
    {"check": "Applicant IDs complete", "passed": pos_clean["SK_ID_CURR"].notna().all()},
    {"check": "Previous-loan IDs complete", "passed": pos_clean["SK_ID_PREV"].notna().all()},
    {"check": "Loan-month keys unique", "passed": not pos_clean.duplicated(["SK_ID_PREV", "MONTHS_BALANCE"]).any()},
    {"check": "No categorical missing values", "passed": pos_clean.select_dtypes(exclude="number").isna().sum().sum() == 0},
    {"check": "No future balance months", "passed": not pos_clean["MONTHS_BALANCE"].gt(0).any()},
    {"check": "No negative installment counts", "passed": not pos_clean[["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]].lt(0).any().any()},
    {"check": "No negative delinquency days", "passed": not pos_clean[["SK_DPD", "SK_DPD_DEF"]].lt(0).any().any()},
    {"check": "No high-missing retained feature", "passed": not (feature_decisions.query("decision == 'Keep'")["missing_rate"] >= MISSING_THRESHOLD).any()},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
])
assert validation_checks["passed"].all(), "At least one POS/cash cleaning check failed."
validation_checks

,check,passed
0,Only project applicants retained,True
1,Applicant IDs complete,True
2,Previous-loan IDs complete,True
3,Loan-month keys unique,True
4,No categorical missing values,True
5,No future balance months,True
6,No negative installment counts,True
7,No negative delinquency days,True
8,No high-missing retained feature,True
9,No infinite numerical values,True


All checks passed.


## Save the clean table and audit reports


In [28]:
cleaning_audit = pd.DataFrame([
    {"rule": "Out-of-scope records removed", "affected": out_of_scope_rows},
    {"rule": "Exact duplicate rows removed", "affected": exact_duplicates},
    {"rule": "Rare status records consolidated", "affected": int(rare_status_mask.sum())},
    {"rule": "Future installments above total flagged", "affected": int(future_above_total.sum())},
    {"rule": "Completed contracts with remaining installments flagged", "affected": int(completed_with_remaining.sum())},
    {"rule": "Active contracts with zero future installments flagged", "affected": int(active_with_zero_future.sum())},
    {"rule": "Features removed by missingness/constant policy", "affected": len(removed_features)},
])
pos_clean.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "pos_cash_feature_decisions.csv", index=False)
cleaning_audit.to_csv(audit_folder / "pos_cash_cleaning_audit.csv", index=False)
validation_checks.to_csv(audit_folder / "pos_cash_cleaning_validation.csv", index=False)
print("Clean POS/cash dataset saved:", output_path)
print("Output rows:", len(pos_clean))
print("Output columns:", pos_clean.shape[1])
print("Unique previous loans:", pos_clean["SK_ID_PREV"].nunique())
print("Unique applicants:", pos_clean["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(pos_clean.select_dtypes(include="number").isna().sum().sum()))

Clean POS/cash dataset saved: /Users/taranveersingh/A-MRP/data/interim/pos_cash_balance_clean.pkl
Output rows: 8543375
Output columns: 14
Unique previous loans: 800337
Unique applicants: 289444
Remaining numerical missing values: 43741


## Main cleaning results

The cleaned POS/cash balance data contains 8,543,375 monthly records for 800,337 previous loans and 289,444 applicants. The IDs are complete and unique, and no duplicate records remain.

Rare contract statuses were grouped together, and unusual values, like future installments being higher than total installments, were kept and flagged instead of removed.

Two features were removed because they ended up constant. The cleaned data has 14 columns. This was the last of the seven source tables to be cleaned.
